# S10 — EA1 | Carga y creación de dataset
**Estudiante:** José David Osorio Gallego  
**Actividad:** Obtener datos de Hugging Face, transformarlos con pandas y documentar el resultado.

Elegí el dataset [Iris de Hugging Face](https://huggingface.co/datasets/scikit-learn/iris) porque permite entender fácilmente las columnas: son medidas de flores y su especie. En este cuaderno separo las responsabilidades: una clase descarga los datos, otra construye el resumen, y aquí hago las transformaciones.

## 1. Preparación y carga

La clase `HFDatasetLoader` se conecta a Hugging Face. Utilizo el split `train` y lo convierto a un DataFrame, sin modificar los datos originales.

In [ ]:
import pandas as pd
from IPython.display import display

from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

In [ ]:
cargador = HFDatasetLoader("scikit-learn/iris", split="train")
df_original = cargador.cargar()

print(f"Datos originales: {df_original.shape[0]} filas y {df_original.shape[1]} columnas")
print("Columnas originales:", ", ".join(df_original.columns))
display(df_original.head())

## 2. Transformaciones del DataFrame

La rúbrica solicita **al menos cuatro transformaciones distintas**. Realizo cinco, mostrando lo que ocurre después de cada una. No cuento `head()` como transformación.

In [ ]:
# Transformación 1: eliminar un identificador que no representa una medición.
df_sin_id = df_original.drop(columns=["Id"])
print("1. Eliminar Id:", df_sin_id.shape)
display(df_sin_id.head(3))

In [ ]:
# Transformación 2: dar nombres descriptivos en español.
df_renombrado = df_sin_id.rename(
    columns={
        "SepalLengthCm": "largo_sepalo_cm",
        "SepalWidthCm": "ancho_sepalo_cm",
        "PetalLengthCm": "largo_petalo_cm",
        "PetalWidthCm": "ancho_petalo_cm",
        "Species": "especie",
    }
)
print("2. Columnas renombradas:", df_renombrado.columns.tolist())
display(df_renombrado.head(3))

In [ ]:
# Transformación 3: conservar flores cuyo pétalo mida más de 1.5 cm.
df_filtrado = df_renombrado.loc[
    df_renombrado["largo_petalo_cm"] > 1.5
].copy().reset_index(drop=True)
print(f"3. Filtrado: {len(df_original)} -> {len(df_filtrado)} registros")
display(df_filtrado.head(3))

In [ ]:
# Transformación 4: calcular una medida aproximada del tamaño del pétalo.
# Largo x ancho es una aproximación rectangular, no el área real de la flor.
df_con_area = df_filtrado.copy()
df_con_area["area_petalo_aprox_cm2"] = (
    df_con_area["largo_petalo_cm"] * df_con_area["ancho_petalo_cm"]
).round(2)
print("4. Columna calculada: area_petalo_aprox_cm2")
display(df_con_area.head(3))

In [ ]:
# Transformación 5: ordenar de mayor a menor área aproximada.
df_final = df_con_area.sort_values(
    by="area_petalo_aprox_cm2", ascending=False
).reset_index(drop=True)

print("5. Ordenar de mayor a menor:", df_final.shape)
display(df_final.head(10))
print("\nCantidad por especie después del filtro:")
print(df_final["especie"].value_counts().to_string())

## 3. Archivo de resumen

La segunda clase recibe **`df_final`**, no el DataFrame original. Así garantizo que el archivo `resumen_iris_transformado.txt` describe las columnas y las filas que quedaron después del procesamiento.

In [ ]:
resumidor = DatasetSummarizer(df_final)
ruta_resumen = resumidor.guardar_txt("resumen_iris_transformado.txt")
texto_resumen = ruta_resumen.read_text(encoding="utf-8")
print(f"Resumen guardado en: {ruta_resumen.resolve()}\n")
print(texto_resumen)

## 4. Comprobación final

Estas verificaciones permiten detectar si por error se resume otro DataFrame o si alguna transformación no se aplicó.

In [ ]:
assert isinstance(df_original, pd.DataFrame)
assert df_original.shape == (150, 6)
assert df_final.shape == (113, 6)
assert "Id" not in df_final.columns
assert "area_petalo_aprox_cm2" in df_final.columns
assert (df_final["largo_petalo_cm"] > 1.5).all()
assert df_final["area_petalo_aprox_cm2"].is_monotonic_decreasing
assert "Filas: 113" in texto_resumen
assert "Columnas: 6" in texto_resumen
assert "area_petalo_aprox_cm2" in texto_resumen

print("Comprobaciones correctas: carga, 5 transformaciones y TXT final.")

## 5. ¿Qué aprendí?

El dataset comenzó con **150 filas y 6 columnas**. Al quitar el identificador, traducir las columnas, filtrar los pétalos de más de 1,5 cm, calcular una medida aproximada y ordenar el resultado, terminé con **113 filas y 6 columnas**. Quedaron 50 registros de *Iris-virginica*, 50 de *Iris-versicolor* y 13 de *Iris-setosa*.

El filtro cambia la proporción de especies, por lo que el conjunto final no tiene la misma distribución que el original. También encontré una fila duplicada exacta en el conjunto transformado: la reporto en el TXT, pero no la elimino porque la actividad solo requiere un procesamiento inicial. El área calculada es una aproximación rectangular para comparar observaciones, no una medición botánica exacta.

La principal ventaja de separar el trabajo en dos clases es que puedo volver a usar la carga y el resumen con otros datasets en las próximas actividades.